# Stage 1 — Fixed-boundary equilibrium

Loads `eq_fixed.h5` and plots 1-D profiles, 2-D cross-sections, and 3-D surface.

In [ ]:
import os
import warnings
from pathlib import Path

# Must be set before any JAX/DESC import
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["JAX_PLATFORM_NAME"] = "cpu"   # older JAX versions use this key
os.environ["JAX_ENABLE_X64"] = "True"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.5"

warnings.filterwarnings("ignore", category=FutureWarning, message=".*pynvml.*")
warnings.filterwarnings("ignore", category=UserWarning, message=".*Unequal number of field periods.*")

import jax
jax.config.update("jax_platforms", "cpu")   # belt-and-suspenders after import

import numpy as np
import matplotlib.pyplot as plt
import plotly.io as pio
from IPython.display import HTML

pio.renderers.render_on_display = False

def show3d(fig):
    return HTML(fig.to_html(include_plotlyjs="cdn", full_html=False))

from desc.equilibrium import Equilibrium
from desc.grid import LinearGrid
from desc.plotting import plot_1d, plot_section, plot_surfaces, plot_3d

HERE = Path(".")
print("JAX backend:", jax.default_backend())

In [ ]:
eq_fixed = Equilibrium.load(str(HERE / "eq_fixed.h5"))
print(f"NFP={eq_fixed.NFP}, L={eq_fixed.L}, M={eq_fixed.M}, N={eq_fixed.N}")

## 1-D profiles

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

plot_1d(eq_fixed, "iota",     ax=axes[0], color="steelblue", label="iota")
plot_1d(eq_fixed, "pressure", ax=axes[1], color="tomato",    label="pressure")
plot_1d(eq_fixed, "current",  ax=axes[2], color="seagreen",  label="current")

axes[0].set_title(r"Rotational transform $\iota(\rho)$")
axes[1].set_title(r"Pressure $p(\rho)$  [Pa]")
axes[2].set_title(r"Enclosed current $I(\rho)$  [A]")
fig.suptitle("Stage 1 — fixed-boundary equilibrium", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

plot_1d(eq_fixed, "|F|",   ax=axes[0], color="purple",     label="|F|")
plot_1d(eq_fixed, "J_phi", ax=axes[1], color="darkorange", label="J_phi")

axes[0].set_title(r"Force residual $|\mathbf{F}|(\rho)$ [N/m³]")
axes[1].set_title(r"Toroidal current density $J_\phi(\rho)$ [A/m²]")
fig.suptitle("Stage 1 — force balance", fontsize=13)
plt.tight_layout()
plt.show()

## 2-D cross-sections

In [ ]:
N_phi = 4
plot_surfaces(
    eq_fixed,
    phi=N_phi,
    rho=np.linspace(0.2, 1.0, 5),
    theta=0,
)
plt.suptitle("Stage 1 — flux surfaces (one field period)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
plot_section(eq_fixed, "|B|", log=False)
plt.suptitle("Stage 1 — |B| [T] (poloidal section)", fontsize=13)
plt.show()

plot_section(eq_fixed, "|F|_normalized", log=True)
plt.suptitle("Stage 1 — normalised |F| (log scale)", fontsize=13)
plt.show()

## 3-D view

In [ ]:
grid_3d = LinearGrid(rho=8, theta=24, zeta=36, NFP=eq_fixed.NFP)
fig = plot_3d(eq_fixed, "|B|", grid=grid_3d, alpha=0.8)
fig.update_layout(title="Stage 1 — fixed-boundary LCFS coloured by |B|")
show3d(fig)